# Data Cleaning

In this notebook, we will perform data cleaning on the raw data files `OSF_LPFSBF20_PID5BF+M_Community.sav` and `PhD_MF_Age-neutrality_Archived.sav`. 

The primary datasets were collected from the OSF repository as detailed in data collection notebook. In this step the data was loaded in using `load_rawdata()` function from the `io.py` module I wrote. The function takes the filename as an argument and returns a pandas DataFrame. It parses the filepath of the desired file using the pathlib module and loads the data into a pandas DataFrame using the pandas `read_spss()` function, since my datasets are SPSS `.sav` files. 

In [ ]:
from exclusions.io import load_rawdata
dfprimary = load_rawdata("PhD_MF_Age-neutrality_Archived.sav")
dfprimary.shape


(398, 64)


Next, I created a column to store the gender variable encoded in the "geslacht" column in the original dataset. I mapped the categorical values in that column to text entries in the new one.

In [49]:
mapping = {
    1 : 'man', 
    2 : 'woman',
    3 : 'non-binary'
} # created a mapping for the gender codes by exploring the file's value labels

dfprimary = dfprimary.rename(columns={"geslacht": "gender_code"}) # translate Dutch "geslacht" to English

dfprimary["gender"] = dfprimary["gender_code"].map(mapping) # map values onto words
print(dfprimary["gender"].value_counts())
print(dfprimary["gender_code"].value_counts()) # confirm that mapping was successful by reviewing value counts


gender
woman         254
man           141
non-binary      3
Name: count, dtype: int64
gender_code
2.0    254
1.0    141
3.0      3
Name: count, dtype: int64


In [ ]:
dfbenchmark = load_rawdata("OSF_LPFSBF20_PID5BF+M_Community.sav")
dfbenchmark.shape

dfbenchmark = dfbenchmark.rename(columns={"Geslacht": "gender_code"}) # translate german "geslacht" to english

dfbenchmark["gender"] = dfbenchmark["gender_code"].map(mapping) # map values onto words
print(dfprimary["gender"].value_counts())
print(dfprimary["gender_code"].value_counts()) # confirm that mapping was successful by reviewing value counts

,gender_code,LT_Cat,CRS_1,CRS_2,CRS_3,CRS_4,CRS_5,LPFS_01,LPFS_02,LPFS_03,...,PID5_irresp,PID5_impuls,PID5_distract,PID5_perfect,PID5_rigid,PID5_orderl,PID5_unbel_exp,PID5_ecc,PID5_perceptdys,gender
0,1.0,1.0,4.0,1.0,4.0,2.0,3.0,3.0,3.0,2.0,...,0.0,0.5,1.0,1.5,1.5,0.0,0.5,0.0,0.0,man
1,1.0,1.0,4.0,1.0,4.0,2.0,3.0,3.0,3.0,3.0,...,1.0,1.0,2.5,0.5,0.0,0.0,0.5,2.0,0.0,man
2,1.0,1.0,4.0,1.0,4.0,2.0,3.0,2.0,3.0,3.0,...,1.0,1.5,2.0,2.5,2.0,1.0,1.0,0.5,0.0,man
3,1.0,1.0,4.0,1.0,4.0,2.0,3.0,2.0,3.0,1.0,...,0.5,1.5,2.0,2.0,1.5,1.5,1.0,1.0,1.0,man
4,1.0,1.0,4.0,1.0,4.0,2.0,3.0,2.0,3.0,2.0,...,0.5,2.0,2.5,0.5,2.0,0.5,2.0,2.0,1.0,man


Some of the scale score column names are set differently in the two datasets. I listed both column names and edited them to converge to one naming convention for ease of analysis.

In [51]:
print(dfprimary.columns)
print(dfbenchmark.columns)

Index(['gender_code', 'LT_Categorie', 'PID5BF_M_01', 'PID5BF_M_02',
       'PID5BF_M_03', 'PID5BF_M_04', 'PID5BF_M_05', 'PID5BF_M_06',
       'PID5BF_M_07', 'PID5BF_M_08', 'PID5BF_M_09', 'PID5BF_M_10',
       'PID5BF_M_11', 'PID5BF_M_12', 'PID5BF_M_13', 'PID5BF_M_14',
       'PID5BF_M_15', 'PID5BF_M_16', 'PID5BF_M_17', 'PID5BF_M_18',
       'PID5BF_M_19', 'PID5BF_M_20', 'PID5BF_M_21', 'PID5BF_M_22',
       'PID5BF_M_23', 'PID5BF_M_24', 'PID5BF_M_25', 'PID5BF_M_26',
       'PID5BF_M_27', 'PID5BF_M_28', 'PID5BF_M_29', 'PID5BF_M_30',
       'PID5BF_M_31', 'PID5BF_M_32', 'PID5BF_M_33', 'PID5BF_M_34',
       'PID5BF_M_35', 'PID5BF_M_36', 'LPFSBF01', 'LPFSBF02', 'LPFSBF03',
       'LPFSBF04', 'LPFSBF05', 'LPFSBF06', 'LPFSBF07', 'LPFSBF08', 'LPFSBF09',
       'LPFSBF10', 'LPFSBF11', 'LPFSBF12', 'CRS1', 'CRS2', 'CRS3', 'CRS4',
       'CRS5', 'Missings_PIDBFM', 'Missings_LPFSBF', 'CRS1_Dummy',
       'CRS2_Dummy', 'CRS3_Dummy', 'CRS4_Dummy', 'CRS5_Dummy', 'CRS_Totaal',
       'CRS_Totaal_Dummy'

In the primary dataset, PID items are coded 1-4, whereas in the benchmark dataset they are coded 0-3. It is ideal to have them both code items 0-3 to match scoring methodology in the literature. I therefore subtracted 1 from each PID item entry in the primary dataset, `dfprimary`.

To-do
*Data structure and checks*

1. Rename everything to English, consistently in both files: age_group (0 = younger, 1 = older) plus a text label, and matching item names.
2. Fix data types: codes become Int64 (pandas' nullable integer), which keeps the NaNs.
3. Check value ranges: every item should be within its expected range, as above. This proves the LPFS "18" problem is only in the labels, not the values.

*CRS (your main variable)*

4. Recompute CRS scores from the answer key (CRS1 = 4, CRS2 = 1, CRS3 = 4, CRS4 = 2, CRS5 = 3). Check them against CRS1_Dummy–CRS5_Dummy, CRS_Totaal and CRS_Totaal_Dummy, and report any disagreements.
5. Create crs_fail (1 = fewer than 3 correct). You should get 57. Keep the CRS items in the data: you need them for sub-question 2. Leave them out of the model features later.

*Missing data (this deserves a real decision)*

6. Here's what the authors' count columns show:

    -  Missings_PIDBFM: 16 people have 19 missing, which fits stopping at item 18. Two more people each have 1 missing.
    - Missings_LPFSBF: 20 people have all 12 missing.
Then:

    - Cross-tab missingness by crs_fail and age group. If dropouts are mostly older or mostly failers, that matters for your research question.
    - Pick a scoring rule: score only complete scales, or allow a small number of missing items and average the rest. Find what Facon et al. or the PID-5-BF+M scoring guidance did, and cite it.
    - Don't silently drop anyone. Count how many each rule excludes.
3. Non-binary respondents (n = 3): the Facon thresholds exist only for men and women. Decide and document whether these 3 are left out of threshold analyses only, or out of everything.

*Scoring (goes in scoring.py)*

8. LPFS: total, Self (items 1–6) and Interpersonal (items 7–12) sums.
9. PID-5-BF+M: 6 domain and 18 facet means on the 0–3 scale. You need the item-to-facet key from the published scoring instructions; take it from the source and cite it. Then test against the benchmark's columns (point 3 above).
10. T-scores and normal / mild / clinical categories from Facon et al. (2026) Table 1, by age group × gender. You'll need to copy the norm values from the paper. I can't supply them without the paper in front of me, and I won't guess numbers.

*Output*

11. Save the cleaned datasets with save_to_csv: one primary and one benchmark file, with matching names.
12. Summary table for the page: N, counts by group, missingness, and CRS failure counts. Show only grouped numbers; this page has to stay free of participant rows.